# 教程: 创建自定义模型组件

**相关模块:** `common/base_model.py`

## 目的
本框架最强大的功能之一是其模块化和可扩展性。用户可以创建自己的模型组件，并将其无缝地集成到`SimulationController`管理的网络中。此教程旨在作为一个分步指南，展示如何创建一个全新的、自定义的模型组件。

此笔记本将：
1.  解释所有模型组件都必须遵循的`BaseModelComponent`接口。
2.  从头开始创建一个新的、简单的组件——`LagAndRoute`，它模拟一个简单的延迟和衰减过程。
3.  将这个新创建的组件放入一个仿真网络中。
4.  运行模拟并验证我们的新组件是否按预期工作。

## 1. `BaseModelComponent` 接口

为了让一个对象能被`SimulationController`识别和管理，它必须继承自`common.base_model.BaseModelComponent`，并实现其定义的抽象方法。这个接口非常简单：

- `__init__(self, name: str)`: 构造函数。必须调用父类的构造函数 `super().__init__(name)`，为组件提供一个唯一的名称。
- `step(self, inflows: dict, dt: float)`: **核心方法**。控制器在每个时间步都会调用这个方法。它负责执行该组件的单步计算逻辑。`inflows`参数是一个字典，包含了所有上游组件的名称和它们的出流量，以及所有在`global_inputs`中定义的全局输入。
- `get_outflow(self) -> float`: 在`step`方法执行完毕后，控制器会调用此方法来获取该组件的输出，以便传递给下游组件。通常，这个方法只返回在`step`中计算出的`self.outflow`属性。


## 2. 创建`LagAndRoute`组件

我们来创建一个名为`LagAndRoute`的组件。它的功能是：
- **延迟 (Lag)**: 将输入流量延迟`lag_steps`个时间步。
- **演算 (Route)**: 使用一个简单的线性水库 `Q_out = Storage * k` 来模拟衰减。

这是一个在水文学中很常见的概念性汇流模型。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
from collections import deque

# 将项目根目录添加到Python路径
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))

from common.base_model import BaseModelComponent
from common.controller import SimulationController

# --- 定义我们的新组件 ---
class LagAndRoute(BaseModelComponent):
    def __init__(self, name: str, lag_steps: int, k: float, input_key: str):
        super().__init__(name)
        if lag_steps < 0:
            raise ValueError("Lag steps cannot be negative.")
        self.k = k # 线性水库出流系数
        self.input_key = input_key # 指定该组件从全局输入中读取哪个key
        
        # 使用双端队列来高效地实现延迟
        self.inflow_buffer = deque([0.0] * lag_steps, maxlen=lag_steps)
        self.storage = 0.0

    def step(self, inflows: dict, dt: float):
        # 1. 从inflows字典中获取该组件关心的输入流量
        # 其他上游组件的来流也会在inflows中，这里为了简单我们只用全局输入
        total_inflow = inflows.get(self.input_key, 0)
        
        # 2. 将当前输入存入缓冲区，并取出延迟后的流量
        self.inflow_buffer.append(total_inflow)
        lagged_inflow = self.inflow_buffer[0] # 最旧的那个就是延迟后的
        
        # 3. 对延迟后的流量进行线性水库演算
        self.storage += lagged_inflow
        routed_outflow = self.storage * self.k
        self.storage -= routed_outflow
        
        # 4. 更新组件的outflow属性
        self.outflow = routed_outflow

print("成功定义新的自定义组件: LagAndRoute")

## 3. 在网络中使用新组件

现在我们可以像使用任何其他内置组件一样来使用`LagAndRoute`。我们创建一个简单的网络，其中一个恒定的源（通过`global_inputs`模拟）流入我们的`LagAndRoute`组件。

In [ ]:
# 实例化我们的新组件
custom_comp = LagAndRoute(name="MyLagComponent", lag_steps=5, k=0.4, input_key='source_inflow')

# 创建控制器并添加组件
controller = SimulationController()
controller.add_component(custom_comp)

# 定义输入
num_steps = 50
inflow_q = np.zeros(num_steps)
inflow_q[5:15] = 100 # 一个方形的入流脉冲

# 使用 global_inputs 将输入直接提供给组件
global_inputs = {
    'source_inflow': inflow_q
}

# 运行模拟并记录历史
history = []
history_generator = controller.run(
    num_steps=num_steps,
    dt=1.0,
    global_inputs=global_inputs
)
for status in history_generator:
    history.append({'MyLagComponent': {'outflow': controller.components['MyLagComponent'].get_outflow()}})

print("使用自定义组件的模拟运行完成。")

## 4. 验证和可视化结果

最后，我们绘制出原始的输入流量和经过我们组件处理后的输出流量，以验证其是否正确地实现了延迟和衰减的效果。

In [ ]:
outflow_history = [h['MyLagComponent']['outflow'] for h in history]
timesteps = np.arange(num_steps)

plt.figure(figsize=(12, 6))
plt.plot(timesteps, inflow_q, 'k--', label='Input to Component')
plt.plot(timesteps, outflow_history, 'b-o', label='Outflow from LagAndRoute')

plt.title('Verification of Custom Model Component')
plt.xlabel('Time Step')
plt.ylabel('Flow')
plt.axvline(5, color='grey', linestyle=':', label='Lag Period (5 steps)')
plt.axvline(10, color='grey', linestyle=':')
plt.legend()
plt.grid(True)
plt.show()

print("从图中可以看到，输出过程线相比于输入过程线，不仅被延迟了5个时间步，而且洪峰被明显削减，过程被拉长，完全符合预期！")